In [ ]:
# Fine-tunes UniFormerV2 on the RHAD dataset and exports its predictions.

In [ ]:
# ============================================================
# CELLULE 1 — Vérification du GPU T4 et du dataset
# ============================================================

import random
import numpy as np
import torch
from pathlib import Path 

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision("high")

print("PyTorch :", torch.__version__)
print("CUDA PyTorch :", torch.version.cuda)
print("CUDA disponible :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA n'est pas disponible. Active un GPU T4 dans Kaggle.")

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)

print("GPU :", gpu_name)
print("Capability :", gpu_capability)
print("Architectures PyTorch :", torch.cuda.get_arch_list())

if gpu_capability < (7, 0):
    raise RuntimeError(
        "Le GPU actuel n'est pas compatible avec cette version de PyTorch. "
        "Sélectionne une Tesla T4 dans Kaggle."
    )

# Test réel sur le GPU
x = torch.randn(1000, 1000, device="cuda")
y = x @ x

print("Calcul GPU réussi :", y.shape)
print("Device :", y.device)

del x, y
torch.cuda.empty_cache()

# Vérification du dataset
BASE = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass/training/Training"
)

assert BASE.exists(), f"Dataset introuvable : {BASE}"

classes = sorted(
    folder.name
    for folder in BASE.iterdir()
    if folder.is_dir()
)

assert len(classes) == 12, (
    f"12 classes attendues, mais {len(classes)} trouvées."
)

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(classes)
}

print("Nombre de classes :", len(classes))
print("Classes :", classes)
print("GPU et dataset correctement configurés.")


In [ ]:
# ============================================================
# CELLULE 2 — Installation et préparation de UniFormerV2
# Ne pas installer OpenMIM, MMCV ou MMAction2
# ============================================================

from pathlib import Path
import os

os.chdir("/kaggle/working")

# Nettoyer uniquement une ancienne copie du dépôt.
!rm -rf /kaggle/working/UniFormerV2
!rm -f /kaggle/working/uniformerv2_model_standalone.py

# Dépôt officiel.
!git clone -q https://github.com/OpenGVLab/UniFormerV2.git

# Dépendances utiles uniquement.
%pip install -q timm decord av huggingface_hub hf_xet scikit-learn matplotlib tqdm

official_file = Path(
    "/kaggle/working/UniFormerV2/slowfast/models/uniformerv2_model.py"
)

standalone_file = Path(
    "/kaggle/working/uniformerv2_model_standalone.py"
)

assert official_file.exists(), (
    f"Fichier officiel introuvable : {official_file}"
)

# Le modèle officiel importe seulement le logger de SlowFast.
# On crée une copie autonome pour éviter d'installer toute la pile SlowFast.
code = official_file.read_text(encoding="utf-8")

code = code.replace(
    "from timm.models.layers import DropPath",
    "from timm.layers import DropPath",
)

code = code.replace(
    "import slowfast.utils.logging as logging",
    "import logging",
)

code = code.replace(
    "logger = logging.get_logger(__name__)",
    "logger = logging.getLogger(__name__)",
)

standalone_file.write_text(
    code,
    encoding="utf-8",
)

assert standalone_file.exists()

print("Dépôt cloné :", Path("/kaggle/working/UniFormerV2").exists())
print("Fichier autonome :", standalone_file)
print("Installation terminée.")


In [ ]:
# ============================================================
# CELLULE 3 — SPLIT IDENTIQUE À VIDEOMAE ET VIDEOMAMBA
# Groupes de 5 clips, seed 42, aucune fuite
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import csv
import random
import re

BASE = Path(
    "/kaggle/input/datasets/"
    "nisrineamalkhnifass/training/Training"
)

ANNO_DIR = Path(
    "/kaggle/working/annotations"
)

ANNO_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mkv",
    ".mov",
    ".webm",
}

SEED = 42
CLIPS_PER_SOURCE = 5

assert BASE.exists(), BASE

classes = sorted(
    folder.name
    for folder in BASE.iterdir()
    if folder.is_dir()
)

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(classes)
}

expected_mapping = {
    "Jumping": 0,
    "child_running": 1,
    "climbing": 2,
    "collesion": 3,
    "electrical_appliances": 4,
    "falling": 5,
    "head_stuck": 6,
    "normal": 7,
    "objects_falling": 8,
    "pulling_curtains": 9,
    "scribbling": 10,
    "sharp_objects": 11,
}

assert class_to_idx == expected_mapping


# ------------------------------------------------------------
# Identifier les groupes de 5 clips
# ------------------------------------------------------------

def extract_group_id(video_path: Path) -> str:
    stem = video_path.stem

    relative_parent = (
        video_path.parent.relative_to(BASE)
    )

    match = re.match(
        r"^(.*?)\s*\((\d+)\)$",
        stem,
    )

    if match:
        prefix = match.group(1).strip()
        clip_number = int(match.group(2))

        block_number = (
            clip_number - 1
        ) // CLIPS_PER_SOURCE

        return (
            f"{relative_parent}/"
            f"{prefix}/"
            f"block_{block_number:04d}"
        )

    # Une vidéo ne suivant pas le format habituel
    # devient son propre groupe.
    return str(
        video_path.relative_to(BASE)
    )


groups_by_class = defaultdict(
    lambda: defaultdict(list)
)

total_videos = 0

for class_name in classes:
    label = class_to_idx[class_name]

    video_files = sorted(
        path
        for path in (
            BASE / class_name
        ).rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in VIDEO_EXTENSIONS
        )
    )

    for video_path in video_files:
        group_id = extract_group_id(
            video_path
        )

        groups_by_class[
            class_name
        ][group_id].append(
            (
                str(video_path),
                label,
            )
        )

        total_videos += 1


assert total_videos == 3248

print("Total vidéos :", total_videos)


# ------------------------------------------------------------
# Créer les splits avec exactement le même générateur
# ------------------------------------------------------------

rng = random.Random(SEED)

train_samples = []
val_samples = []
test_samples = []

train_groups = set()
val_groups = set()
test_groups = set()

split_manifest = []


for class_name in classes:
    class_groups = list(
        groups_by_class[
            class_name
        ].items()
    )

    rng.shuffle(class_groups)

    number_of_groups = len(
        class_groups
    )

    assert number_of_groups >= 3

    number_test = max(
        1,
        round(
            number_of_groups * 0.10
        ),
    )

    number_val = max(
        1,
        round(
            number_of_groups * 0.10
        ),
    )

    while (
        number_test + number_val
        >= number_of_groups
    ):
        if number_test > 1:
            number_test -= 1
        elif number_val > 1:
            number_val -= 1
        else:
            break

    test_part = class_groups[
        :number_test
    ]

    val_part = class_groups[
        number_test:
        number_test + number_val
    ]

    train_part = class_groups[
        number_test + number_val:
    ]

    split_definitions = [
        (
            "train",
            train_part,
            train_samples,
            train_groups,
        ),
        (
            "val",
            val_part,
            val_samples,
            val_groups,
        ),
        (
            "test",
            test_part,
            test_samples,
            test_groups,
        ),
    ]

    for (
        split_name,
        selected_groups,
        samples,
        group_set,
    ) in split_definitions:

        for (
            group_id,
            group_samples,
        ) in selected_groups:

            group_set.add(group_id)

            samples.extend(
                group_samples
            )

            split_manifest.append(
                {
                    "split": split_name,
                    "class_name": class_name,
                    "label": class_to_idx[
                        class_name
                    ],
                    "group_id": group_id,
                    "number_of_clips": len(
                        group_samples
                    ),
                }
            )


# Même mélange que dans VideoMAE
rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)


# ------------------------------------------------------------
# Vérifications
# ------------------------------------------------------------

assert train_groups.isdisjoint(
    val_groups
)

assert train_groups.isdisjoint(
    test_groups
)

assert val_groups.isdisjoint(
    test_groups
)

assert len(train_samples) == 2578
assert len(val_samples) == 335
assert len(test_samples) == 335

assert (
    len(train_samples)
    + len(val_samples)
    + len(test_samples)
    == 3248
)


# ------------------------------------------------------------
# Sauvegarder les annotations
# ------------------------------------------------------------

def write_samples_csv(
    destination,
    samples,
):
    with destination.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as file:

        writer = csv.writer(file)

        for path, label in samples:
            writer.writerow(
                [
                    path,
                    int(label),
                ]
            )


write_samples_csv(
    ANNO_DIR / "train.csv",
    train_samples,
)

write_samples_csv(
    ANNO_DIR / "val.csv",
    val_samples,
)

write_samples_csv(
    ANNO_DIR / "test.csv",
    test_samples,
)


with (
    ANNO_DIR
    / "label_mapping.csv"
).open(
    "w",
    newline="",
    encoding="utf-8",
) as file:

    writer = csv.writer(file)

    writer.writerow(
        [
            "label",
            "class_name",
        ]
    )

    for class_name, label in (
        class_to_idx.items()
    ):
        writer.writerow(
            [
                label,
                class_name,
            ]
        )


with (
    ANNO_DIR
    / "split_manifest.csv"
).open(
    "w",
    newline="",
    encoding="utf-8",
) as file:

    fieldnames = [
        "split",
        "class_name",
        "label",
        "group_id",
        "number_of_clips",
    ]

    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames,
    )

    writer.writeheader()
    writer.writerows(
        split_manifest
    )


# ------------------------------------------------------------
# Afficher les distributions
# ------------------------------------------------------------

def print_distribution(
    split_name,
    samples,
):
    counts = Counter(
        label
        for _, label
        in samples
    )

    print(f"\n{split_name} :")

    for label, class_name in enumerate(
        classes
    ):
        print(
            f"  {class_name:<24} "
            f"{counts[label]:>4}"
        )


print(
    "\n"
    f"Train : {len(train_samples)} | "
    f"Validation : {len(val_samples)} | "
    f"Test : {len(test_samples)}"
)

print(
    "Fuites train/validation :",
    len(
        train_groups
        & val_groups
    ),
)

print(
    "Fuites train/test       :",
    len(
        train_groups
        & test_groups
    ),
)

print(
    "Fuites validation/test  :",
    len(
        val_groups
        & test_groups
    ),
)

print_distribution(
    "TRAIN",
    train_samples,
)

print_distribution(
    "VALIDATION",
    val_samples,
)

print_distribution(
    "TEST",
    test_samples,
)

print(
    "\nSPLIT IDENTIQUE À VIDEOMAE "
    "ET VIDEOMAMBA — AUCUNE FUITE"
)

In [ ]:
# ============================================================
# CELLULE 4 — Téléchargement du checkpoint UniFormerV2-B/16
# Préentraînement Kinetics-400 + Kinetics-710
# ============================================================

from pathlib import Path
from huggingface_hub import hf_hub_download

CKPT_DIR = Path(
    "/kaggle/working/checkpoints"
)

CKPT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_FILENAME = (
    "k400+k710_uniformerv2_b16_8x224.pyth"
)

expected_path = (
    CKPT_DIR / CHECKPOINT_FILENAME
)

if (
    expected_path.exists()
    and expected_path.stat().st_size > 400 * 1024**2
):
    CKPT_PATH = expected_path
    print("Checkpoint déjà présent.")

else:
    download_errors = []

    # Miroir public du checkpoint publié pour UniFormerV2.
    download_candidates = [
        (
            "Andy1621/uniformerv2",
            CHECKPOINT_FILENAME,
        ),
        (
            "OpenGVLab/InternVideo1.0",
            (
                "uniformerv2/k400/"
                "k400+k710_uniformerv2_b16_8x224.pyth"
            ),
        ),
    ]

    CKPT_PATH = None

    for repo_id, filename in download_candidates:
        try:
            print(
                f"Tentative de téléchargement : "
                f"{repo_id}/{filename}"
            )

            downloaded_path = hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                local_dir=str(CKPT_DIR),
            )

            downloaded_path = Path(downloaded_path)

            # Le dépôt officiel place éventuellement le fichier
            # dans un sous-dossier. On le copie vers un chemin stable.
            if downloaded_path != expected_path:
                expected_path.write_bytes(
                    downloaded_path.read_bytes()
                )

            CKPT_PATH = expected_path
            break

        except Exception as error:
            download_errors.append(
                f"{repo_id}: {type(error).__name__}: {error}"
            )

    if CKPT_PATH is None:
        raise RuntimeError(
            "Impossible de télécharger le checkpoint.\n\n"
            + "\n\n".join(download_errors)
            + "\n\nVérifie que l'option Internet est activée dans Kaggle."
        )

CKPT_PATH = Path(CKPT_PATH)

size_mb = (
    CKPT_PATH.stat().st_size
    / 1024**2
)

print("Checkpoint :", CKPT_PATH)
print(f"Taille : {size_mb:.1f} Mo")

assert size_mb > 400, (
    "Le checkpoint est vide ou incomplet. "
    "Supprime-le puis relance cette cellule."
)

print("Checkpoint prêt.")


In [ ]:
# ============================================================
# CELLULE 5 — Dataset vidéo et DataLoaders
# Sortie d'une vidéo : (C, T, H, W)
# Sortie d'un batch : (B, C, T, H, W)
# ============================================================

import csv
import random
import warnings
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import torchvision.transforms.functional as TF
from PIL import Image
from decord import VideoReader, cpu

NUM_FRAMES = 8
IMG_SIZE = 224
BATCH_SIZE = 2
NUM_WORKERS = 2

ANNO_DIR = Path(
    "/kaggle/working/annotations"
)


class VideoDataset(Dataset):
    MEAN = [0.45, 0.45, 0.45]
    STD = [0.225, 0.225, 0.225]

    def __init__(self, csv_path, mode="train"):
        self.csv_path = Path(csv_path)
        self.mode = mode
        self.samples = []

        if mode not in {"train", "val", "test"}:
            raise ValueError("mode doit être train, val ou test.")

        with self.csv_path.open(newline="", encoding="utf-8") as file:
            for row in csv.reader(file):
                if len(row) >= 2:
                    self.samples.append((row[0], int(row[1])))

        if not self.samples:
            raise RuntimeError(f"Aucune vidéo trouvée dans {self.csv_path}")

    def __len__(self):
        return len(self.samples)

    def _sample_indices(self, total_frames):
        if total_frames <= 0:
            raise RuntimeError("La vidéo ne contient aucune frame.")

        if total_frames < NUM_FRAMES:
            return np.linspace(0, total_frames - 1, NUM_FRAMES).astype(np.int64)

        boundaries = np.linspace(0, total_frames, NUM_FRAMES + 1).astype(np.int64)
        indices = []

        for left, right in zip(boundaries[:-1], boundaries[1:]):
            right = max(left + 1, min(right, total_frames))

            if self.mode == "train":
                index = random.randrange(left, right)
            else:
                index = min(total_frames - 1, (left + right - 1) // 2)

            indices.append(index)

        return np.asarray(indices, dtype=np.int64)

    def _read_video(self, video_path):
        try:
            video_reader = VideoReader(video_path, ctx=cpu(0), num_threads=1)
            indices = self._sample_indices(len(video_reader))
            frames = video_reader.get_batch(indices).asnumpy()
            return [Image.fromarray(frame).convert("RGB") for frame in frames]

        except Exception as error:
            warnings.warn(f"Lecture impossible : {video_path} | {error}")
            return [Image.new("RGB", (IMG_SIZE, IMG_SIZE)) for _ in range(NUM_FRAMES)]

    def _transform_train(self, images):
        i, j, h, w = transforms.RandomResizedCrop.get_params(
            images[0], scale=(0.5, 1.0), ratio=(0.75, 1.3333)
        )

        images = [
            TF.resized_crop(
                image, i, j, h, w, (IMG_SIZE, IMG_SIZE),
                interpolation=transforms.InterpolationMode.BICUBIC,
            )
            for image in images
        ]

        if random.random() < 0.5:
            images = [TF.hflip(image) for image in images]

        if random.random() < 0.8:
            brightness = random.uniform(0.8, 1.2)
            contrast = random.uniform(0.8, 1.2)
            saturation = random.uniform(0.8, 1.2)

            images = [
                TF.adjust_saturation(
                    TF.adjust_contrast(
                        TF.adjust_brightness(image, brightness),
                        contrast,
                    ),
                    saturation,
                )
                for image in images
            ]

        return images

    def _transform_eval(self, images):
        images = [
            TF.resize(image, 256, interpolation=transforms.InterpolationMode.BICUBIC)
            for image in images
        ]
        return [TF.center_crop(image, IMG_SIZE) for image in images]

    def __getitem__(self, index):
        video_path, label = self.samples[index]
        images = self._read_video(video_path)

        if self.mode == "train":
            images = self._transform_train(images)
        else:
            images = self._transform_eval(images)

        frame_tensors = [
            TF.normalize(TF.to_tensor(image), self.MEAN, self.STD)
            for image in images
        ]

        video = torch.stack(frame_tensors, dim=1)
        return video, label


train_dataset = VideoDataset(ANNO_DIR / "train.csv", mode="train")
val_dataset = VideoDataset(ANNO_DIR / "val.csv", mode="val")
test_dataset = VideoDataset(ANNO_DIR / "test.csv", mode="test")


# ============================================================
# WeightedRandomSampler — rééchantillonnage des classes rares
# ============================================================

train_labels = [label for _, label in train_dataset.samples]
class_counts = Counter(train_labels)

print("Distribution du train set :")
for label in range(len(classes)):
    print(f"  {classes[label]:<24} {class_counts[label]:>4}")

sample_weights = [
    1.0 / class_counts[label]
    for _, label in train_dataset.samples
]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True,
)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(SEED)

loader_common = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": True,
    "persistent_workers": NUM_WORKERS > 0,
    "worker_init_fn": seed_worker,
    "generator": generator,
}

# shuffle=False car le sampler gère déjà l'ordre
train_loader = DataLoader(
    train_dataset,
    sampler=sampler,
    drop_last=True,
    **loader_common,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    drop_last=False,
    **loader_common,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    drop_last=False,
    **loader_common,
)

videos, labels = next(iter(train_loader))

print("\nBatch vidéo :", videos.shape)
print("Labels :", labels)
print("Train :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test :", len(test_dataset))

assert videos.ndim == 5
assert videos.shape[1:] == (3, NUM_FRAMES, IMG_SIZE, IMG_SIZE)

print("Format correct : (B, C, T, H, W)")

In [ ]:
# ============================================================
# CELLULE 6 — UniFormerV2 avec fine-tuning partiel
# ============================================================

import importlib.util
from pathlib import Path

import torch
import torch.nn as nn


NUM_CLASSES = len(classes)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_FILE = Path(
    "/kaggle/working/uniformerv2_model_standalone.py"
)

assert MODEL_FILE.exists(), (
    "Le fichier du modèle est absent. Relance la cellule 2."
)

assert Path(CKPT_PATH).exists(), (
    "Le checkpoint préentraîné est absent. Relance la cellule 4."
)


spec = importlib.util.spec_from_file_location(
    "uniformerv2_model_standalone",
    MODEL_FILE,
)

uniformerv2_module = importlib.util.module_from_spec(spec)

spec.loader.exec_module(uniformerv2_module)


model = uniformerv2_module.uniformerv2_b16(
    pretrained=False,
    use_checkpoint=False,
    checkpoint_num=[0],
    t_size=NUM_FRAMES,
    dw_reduction=1.5,
    backbone_drop_path_rate=0.0,
    temporal_downsample=False,
    no_lmhra=True,
    double_lmhra=True,
    return_list=[8, 9, 10, 11],
    n_layers=4,
    n_dim=768,
    n_head=12,
    mlp_factor=4.0,
    drop_path_rate=0.0,
    mlp_dropout=[0.5, 0.5, 0.5, 0.5],
    cls_dropout=0.5,
    num_classes=400,
    frozen=False,
)


def find_state_dict(checkpoint):

    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Le checkpoint n'est pas un dictionnaire."
        )

    possible_keys = [
        "model_state",
        "model_state_dict",
        "state_dict",
        "model",
        "module",
    ]

    for key in possible_keys:

        candidate = checkpoint.get(key)

        if isinstance(candidate, dict):
            return find_state_dict(candidate)

    tensor_items = {
        key: value
        for key, value in checkpoint.items()
        if isinstance(key, str) and torch.is_tensor(value)
    }

    if tensor_items:
        return tensor_items

    raise RuntimeError(
        "Aucun state_dict trouvé dans le checkpoint."
    )


raw_checkpoint = torch.load(
    CKPT_PATH,
    map_location="cpu",
    weights_only=False,
)

checkpoint_state = find_state_dict(
    raw_checkpoint
)


clean_state = {}

for key, value in checkpoint_state.items():

    new_key = key
    prefix_removed = True

    while prefix_removed:

        prefix_removed = False

        for prefix in (
            "module.",
            "model.",
            "backbone.",
        ):

            if new_key.startswith(prefix):

                new_key = new_key[len(prefix):]
                prefix_removed = True

    clean_state[new_key] = value


model_state = model.state_dict()

compatible_state = {
    key: value
    for key, value in clean_state.items()
    if (
        key in model_state
        and model_state[key].shape == value.shape
    )
}

load_result = model.load_state_dict(
    compatible_state,
    strict=False,
)

print(
    "Tenseurs dans le checkpoint :",
    len(clean_state),
)

print(
    "Tenseurs préentraînés chargés :",
    len(compatible_state),
)

print(
    "Clés manquantes :",
    len(load_result.missing_keys),
)

print(
    "Clés inattendues :",
    len(load_result.unexpected_keys),
)

assert len(compatible_state) > 100, (
    "Trop peu de poids ont été chargés."
)


old_classifier = model.transformer.proj[-1]

model.transformer.proj[-1] = nn.Linear(
    old_classifier.in_features,
    NUM_CLASSES,
)

nn.init.trunc_normal_(
    model.transformer.proj[-1].weight,
    std=0.02,
)

nn.init.zeros_(
    model.transformer.proj[-1].bias
)


for parameter in model.parameters():
    parameter.requires_grad = False


def unfreeze_module(module):

    for parameter in module.parameters():
        parameter.requires_grad = True


for block_index in [-2, -1]:

    unfreeze_module(
        model.transformer.resblocks[block_index]
    )


for block_index in [-2, -1]:

    unfreeze_module(
        model.transformer.dpe[block_index]
    )

    unfreeze_module(
        model.transformer.dec[block_index]
    )


unfreeze_module(
    model.transformer.proj
)


model.transformer.temporal_cls_token.requires_grad = True

if hasattr(model.transformer, "balance"):
    model.transformer.balance.requires_grad = True


model = model.to(DEVICE)


total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

frozen_parameters = (
    total_parameters - trainable_parameters
)

print(
    "\nParamètres totaux :",
    f"{total_parameters / 1e6:.2f} M",
)

print(
    "Paramètres entraînables :",
    f"{trainable_parameters / 1e6:.2f} M",
)

print(
    "Paramètres gelés :",
    f"{frozen_parameters / 1e6:.2f} M",
)

print(
    "Pourcentage entraînable :",
    f"{100 * trainable_parameters / total_parameters:.2f} %",
)

assert trainable_parameters > 0
assert trainable_parameters < total_parameters


model.eval()

with torch.inference_mode():

    dummy_video = torch.randn(
        1,
        3,
        NUM_FRAMES,
        IMG_SIZE,
        IMG_SIZE,
        device=DEVICE,
    )

    with torch.amp.autocast(
        device_type="cuda",
        enabled=DEVICE.type == "cuda",
        dtype=torch.float16,
    ):

        dummy_output = model(dummy_video)


print(
    "\nSortie du modèle :",
    dummy_output.shape,
)

assert dummy_output.shape == (
    1,
    NUM_CLASSES,
)

del dummy_video
del dummy_output

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


FRESH_MODEL_PATH = Path(
    "/kaggle/working/"
    "uniformerv2_fresh_initial_state.pth"
)

torch.save(
    model.state_dict(),
    FRESH_MODEL_PATH,
)

print(
    "État initial frais sauvegardé :",
    FRESH_MODEL_PATH,
)

In [ ]:
# ============================================================
# CELLULE 7 — FINE-TUNING PARTIEL + EARLY STOPPING
# (avec pondération de classe pour compenser le déséquilibre)
# ============================================================

import json
import shutil
import time
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn

from tqdm.auto import tqdm
from sklearn.metrics import f1_score


EPOCHS = 40
BACKBONE_LR = 1e-6
HEAD_LR = 2e-5
WEIGHT_DECAY = 0.05
ACCUMULATION_STEPS = 2
WARMUP_EPOCHS = 3
PATIENCE = 5
MIN_IMPROVEMENT = 0.0001

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "uniformerv2_output_partial_finetuning"
)

LATEST_CHECKPOINT_PATH = OUTPUT_DIR / "latest_checkpoint.pth"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model.pth"
HISTORY_PATH = OUTPUT_DIR / "history.json"

RESET_TRAINING = True

if RESET_TRAINING and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    print("Ancien essai supprimé.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FRESH_MODEL_PATH = Path(
    "/kaggle/working/"
    "uniformerv2_fresh_initial_state.pth"
)

if RESET_TRAINING:
    if not FRESH_MODEL_PATH.exists():
        raise FileNotFoundError("Relance d'abord la cellule 6.")

    fresh_model_state = torch.load(
        FRESH_MODEL_PATH,
        map_location="cpu",
        weights_only=True,
    )
    model.load_state_dict(fresh_model_state, strict=True)
    model = model.to(DEVICE)
    print("Poids du modèle restaurés à l'état initial.")


head_parameters = [
    parameter
    for parameter in model.transformer.proj.parameters()
    if parameter.requires_grad
]

head_parameter_ids = {id(parameter) for parameter in head_parameters}

backbone_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad and id(parameter) not in head_parameter_ids
]

number_backbone_parameters = sum(p.numel() for p in backbone_parameters)
number_head_parameters = sum(p.numel() for p in head_parameters)

print(f"Paramètres backbone optimisés : {number_backbone_parameters / 1e6:.2f} M")
print(f"Paramètres tête optimisés : {number_head_parameters / 1e6:.2f} M")

assert number_backbone_parameters > 0
assert number_head_parameters > 0

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_parameters, "lr": BACKBONE_LR},
        {"params": head_parameters, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# PONDÉRATION DES CLASSES
# ============================================================

train_labels = [label for _, label in train_dataset.samples]
class_counts_train = Counter(train_labels)
total_samples = len(train_labels)

class_weights = torch.tensor(
    [
        total_samples / (NUM_CLASSES * class_counts_train[label])
        for label in range(NUM_CLASSES)
    ],
    dtype=torch.float32,
).to(DEVICE)

print("\nPoids par classe :")
for label in range(NUM_CLASSES):
    print(f"  {classes[label]:<24} {class_weights[label]:.3f}")

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1,
)


warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.2, end_factor=1.0, total_iters=WARMUP_EPOCHS,
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=1e-7,
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_EPOCHS],
)

AMP_ENABLED = (DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

start_epoch = 0
best_validation_macro_f1 = 0.0
best_validation_accuracy = 0.0
best_validation_loss = float("inf")
epochs_without_improvement = 0
history = []

if not RESET_TRAINING and LATEST_CHECKPOINT_PATH.exists():
    resume_checkpoint = torch.load(
        LATEST_CHECKPOINT_PATH, map_location="cpu", weights_only=False,
    )
    model.load_state_dict(resume_checkpoint["model_state_dict"])
    optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(resume_checkpoint["scheduler_state_dict"])

    if "scaler_state_dict" in resume_checkpoint:
        scaler.load_state_dict(resume_checkpoint["scaler_state_dict"])

    start_epoch = int(resume_checkpoint["epoch"]) + 1
    best_validation_macro_f1 = float(resume_checkpoint.get("best_validation_macro_f1", 0.0))
    best_validation_accuracy = float(resume_checkpoint.get("best_validation_accuracy", 0.0))
    best_validation_loss = float(resume_checkpoint.get("best_validation_loss", float("inf")))
    epochs_without_improvement = int(resume_checkpoint.get("epochs_without_improvement", 0))
    history = list(resume_checkpoint.get("history", []))

    print(f"Reprise à l'epoch {start_epoch + 1}/{EPOCHS}")
else:
    print("Nouvel entraînement.")


def run_validation():
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        progress = tqdm(
            val_loader,
            desc="Validation",
            leave=False,
        )

        for videos, labels in progress:
            videos = videos.to(
                DEVICE,
                non_blocking=True,
            )
            labels = labels.to(
                DEVICE,
                non_blocking=True,
            )

            with torch.amp.autocast(
                device_type="cuda",
                enabled=AMP_ENABLED,
                dtype=torch.float16,
            ):
                logits = model(videos)
                loss = criterion(
                    logits,
                    labels,
                )

            predictions = logits.argmax(
                dim=1
            )

            batch_size = labels.size(0)
            total_loss += (
                loss.item() * batch_size
            )
            total_correct += (
                predictions == labels
            ).sum().item()
            total_samples += batch_size

            all_labels.extend(
                labels.cpu().tolist()
            )
            all_predictions.extend(
                predictions.cpu().tolist()
            )

    validation_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0,
    )

    return (
        total_loss / total_samples,
        total_correct / total_samples,
        validation_macro_f1,
    )


if RESET_TRAINING and start_epoch == 0:
    (
        initial_validation_loss,
        initial_validation_accuracy,
        initial_validation_macro_f1,
    ) = run_validation()

    print(
        f"\nAvant entraînement | "
        f"val_loss={initial_validation_loss:.4f} | "
        f"val_acc={100 * initial_validation_accuracy:.2f}% | "
        f"val_macro_f1={initial_validation_macro_f1:.4f}"
    )

    if initial_validation_accuracy > 0.50:
        print(
            "\nALERTE : accuracy anormalement "
            "élevée avant l'entraînement."
        )


for epoch in range(start_epoch, EPOCHS):

    epoch_start_time = time.time()
    model.train()
    optimizer.zero_grad(set_to_none=True)

    train_loss_sum = 0.0
    train_correct = 0
    train_samples = 0

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )

    for step, (videos, labels) in progress:
        videos = videos.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda", enabled=AMP_ENABLED, dtype=torch.float16,
        ):
            logits = model(videos)
            loss = criterion(logits, labels)
            loss_for_backward = loss / ACCUMULATION_STEPS

        scaler.scale(loss_for_backward).backward()

        should_update = (
            (step + 1) % ACCUMULATION_STEPS == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=5.0,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        predictions = logits.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_samples += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{100 * train_correct / train_samples:.2f}%",
            backbone_lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            head_lr=f"{optimizer.param_groups[1]['lr']:.2e}",
        )

    train_loss = train_loss_sum / train_samples
    train_accuracy = train_correct / train_samples

    (
        validation_loss,
        validation_accuracy,
        validation_macro_f1,
    ) = run_validation()

    improved_macro_f1 = (
        validation_macro_f1
        > best_validation_macro_f1 + MIN_IMPROVEMENT
    )
    same_macro_better_accuracy = (
        abs(
            validation_macro_f1
            - best_validation_macro_f1
        ) <= MIN_IMPROVEMENT
        and validation_accuracy
        > best_validation_accuracy + MIN_IMPROVEMENT
    )
    same_macro_accuracy_better_loss = (
        abs(
            validation_macro_f1
            - best_validation_macro_f1
        ) <= MIN_IMPROVEMENT
        and abs(
            validation_accuracy
            - best_validation_accuracy
        ) <= MIN_IMPROVEMENT
        and validation_loss < best_validation_loss
    )
    improved = (
        improved_macro_f1
        or same_macro_better_accuracy
        or same_macro_accuracy_better_loss
    )

    if improved:
        best_validation_macro_f1 = validation_macro_f1
        best_validation_accuracy = validation_accuracy
        best_validation_loss = validation_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "classes": classes,
                "class_to_idx": class_to_idx,
                "num_classes": NUM_CLASSES,
                "num_frames": NUM_FRAMES,
                "img_size": IMG_SIZE,
                "best_validation_macro_f1": best_validation_macro_f1,
                "best_validation_accuracy": best_validation_accuracy,
                "best_validation_loss": best_validation_loss,
                "architecture": "UniFormerV2-B/16",
                "training_strategy": "partial fine-tuning",
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    scheduler.step()
    epoch_duration = time.time() - epoch_start_time

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
        "validation_macro_f1": validation_macro_f1,
        "backbone_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"],
        "duration_seconds": epoch_duration,
        "epochs_without_improvement": epochs_without_improvement,
    }
    history.append(epoch_result)

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_validation_macro_f1": best_validation_macro_f1,
            "best_validation_accuracy": best_validation_accuracy,
            "best_validation_loss": best_validation_loss,
            "epochs_without_improvement": epochs_without_improvement,
            "history": history,
            "classes": classes,
            "class_to_idx": class_to_idx,
            "num_classes": NUM_CLASSES,
            "num_frames": NUM_FRAMES,
            "img_size": IMG_SIZE,
        },
        LATEST_CHECKPOINT_PATH,
    )

    HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")

    print(
        f"\nEpoch {epoch + 1:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_accuracy * 100:.2f}% | "
        f"val_loss={validation_loss:.4f} | "
        f"val_acc={validation_accuracy * 100:.2f}% | "
        f"val_macro_f1={validation_macro_f1:.4f} | "
        f"durée={epoch_duration / 60:.1f} min"
        + (" | BEST" if improved else "")
    )

    print(f"Epochs sans amélioration : {epochs_without_improvement}/{PATIENCE}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEARLY STOPPING : aucune amélioration pendant {PATIENCE} epochs.")
        break


print(f"\nMeilleur validation macro F1 : {best_validation_macro_f1:.4f}")
print(f"Meilleure validation accuracy : {best_validation_accuracy * 100:.2f}%")
print(f"Meilleure validation loss : {best_validation_loss:.4f}")
print(f"Checkpoint de reprise : {LATEST_CHECKPOINT_PATH}")
print(f"Meilleur modèle : {BEST_MODEL_PATH}")

In [ ]:
# ============================================================
# CELLULE 8bis — Évaluation finale sur le TEST SET
# ============================================================

import json
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import csv

# Charger le meilleur modèle sauvegardé
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(best_checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

print(
    "Best model chargé | "
    f"epoch={best_checkpoint['epoch'] + 1} | "
    f"val_acc={100 * best_checkpoint['best_validation_accuracy']:.2f}%"
)

all_preds = []
all_labels = []
all_paths = [path for path, _ in test_dataset.samples]

with torch.inference_mode():
    for videos, labels in test_loader:
        videos = videos.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=AMP_ENABLED,
            dtype=torch.float16,
        ):
            logits = model(videos)

        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

test_accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

print(f"\nTest accuracy : {100 * test_accuracy:.2f}%")

# Rapport de classification détaillé (precision/recall/f1 par classe)
report_text = classification_report(
    all_labels, all_preds, target_names=classes, digits=3
)
print("\n" + report_text)

report_dict = classification_report(
    all_labels, all_preds, target_names=classes, digits=3, output_dict=True
)

(OUTPUT_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
(OUTPUT_DIR / "classification_report.json").write_text(
    json.dumps(report_dict, indent=2), encoding="utf-8"
)

# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=90)
ax.set_yticklabels(classes)
ax.set_xlabel("Prédit")
ax.set_ylabel("Vrai")
ax.set_title("Matrice de confusion — Test set")

for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

# CSV des prédictions individuelles
with (OUTPUT_DIR / "test_predictions.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["video_path", "true_label", "true_class", "pred_label", "pred_class", "correct"])
    for path, true_l, pred_l in zip(all_paths, all_labels, all_preds):
        writer.writerow([
            path, true_l, classes[true_l], pred_l, classes[pred_l], int(true_l == pred_l)
        ])

print("\nFichiers sauvegardés dans", OUTPUT_DIR)

In [ ]:
# ============================================================
# CELLULE 9 — Courbes d'entraînement
# ============================================================

import json
import matplotlib.pyplot as plt

assert HISTORY_PATH.exists(), (
    "Le fichier history.json est absent."
)

history = json.loads(
    HISTORY_PATH.read_text(
        encoding="utf-8"
    )
)

epochs = [
    item["epoch"]
    for item in history
]

train_losses = [
    item["train_loss"]
    for item in history
]

validation_losses = [
    item["validation_loss"]
    for item in history
]

train_accuracies = [
    100 * item["train_accuracy"]
    for item in history
]

validation_accuracies = [
    100 * item["validation_accuracy"]
    for item in history
]

# Courbe des pertes.
figure_loss, axis_loss = plt.subplots(
    figsize=(8, 5)
)

axis_loss.plot(
    epochs,
    train_losses,
    label="Train",
)

axis_loss.plot(
    epochs,
    validation_losses,
    label="Validation",
)

axis_loss.set_xlabel("Epoch")
axis_loss.set_ylabel("Loss")
axis_loss.set_title("Loss — UniFormerV2-B/16")
axis_loss.grid(True, alpha=0.3)
axis_loss.legend()

figure_loss.tight_layout()

loss_path = (
    OUTPUT_DIR / "loss_curve.png"
)

figure_loss.savefig(
    loss_path,
    dpi=160,
    bbox_inches="tight",
)

plt.show()

# Courbe des accuracies.
figure_accuracy, axis_accuracy = plt.subplots(
    figsize=(8, 5)
)

axis_accuracy.plot(
    epochs,
    train_accuracies,
    label="Train",
)

axis_accuracy.plot(
    epochs,
    validation_accuracies,
    label="Validation",
)

axis_accuracy.set_xlabel("Epoch")
axis_accuracy.set_ylabel("Accuracy (%)")
axis_accuracy.set_title("Accuracy — UniFormerV2-B/16")
axis_accuracy.grid(True, alpha=0.3)
axis_accuracy.legend()

figure_accuracy.tight_layout()

accuracy_path = (
    OUTPUT_DIR / "accuracy_curve.png"
)

figure_accuracy.savefig(
    accuracy_path,
    dpi=160,
    bbox_inches="tight",
)

plt.show()

print("Courbe loss :", loss_path)
print("Courbe accuracy :", accuracy_path)


In [ ]:
# ============================================================
# CELLULE 10 — Préparer les fichiers à télécharger
# ============================================================

import shutil
from pathlib import Path

RESULTS_DIR = Path(
    "/kaggle/working/uniformerv2_results_light"
)

if RESULTS_DIR.exists():
    shutil.rmtree(
        RESULTS_DIR
    )

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

files_to_copy = [
    ANNO_DIR / "label_mapping.csv",
    ANNO_DIR / "split_manifest.csv",
    OUTPUT_DIR / "history.json",
    OUTPUT_DIR / "classification_report.txt",
    OUTPUT_DIR / "classification_report.json",
    OUTPUT_DIR / "test_predictions.csv",
    OUTPUT_DIR / "confusion_matrix.png",
    OUTPUT_DIR / "loss_curve.png",
    OUTPUT_DIR / "accuracy_curve.png",
]

for source in files_to_copy:
    if source.exists():
        shutil.copy2(
            source,
            RESULTS_DIR / source.name,
        )

zip_path = shutil.make_archive(
    "/kaggle/working/uniformerv2_results_light",
    "zip",
    root_dir=RESULTS_DIR,
)

print("Résultats légers :", zip_path)
print("Meilleur modèle :", BEST_MODEL_PATH)
print("Checkpoint de reprise :", LATEST_CHECKPOINT_PATH)

print(
    "\nTélécharge au minimum :\n"
    "1. best_model.pth pour l'évaluation/inférence\n"
    "2. latest_checkpoint.pth pour reprendre l'entraînement\n"
    "3. uniformerv2_results_light.zip pour les rapports et graphiques"
)


# Prédictions finales et sauvegarde

Après la fin de l'entraînement, exécuter **UF1 à UF6 dans l'ordre**.

- **UF1** charge le meilleur checkpoint.
- **UF2** crée `uniformerv2_validation_predictions.csv` et `uniformerv2_test_predictions.csv`.
- **UF3** crée les matrices de confusion.
- **UF4** crée le résumé et l'archive légère.
- **UF5** sauvegarde le meilleur modèle et tous les résultats dans un dataset Kaggle.
- **UF6** vérifie que tous les fichiers nécessaires à `comparison_and_AT` sont présents et valides.


In [ ]:
# ============================================================
# UF1 — CHARGER LE MEILLEUR CHECKPOINT UNIFORMERV2
# ============================================================

from pathlib import Path
import torch

# Vérifier que les cellules précédentes ont bien été exécutées.
required_variables = [
    "model",
    "val_loader",
    "test_loader",
]

for variable_name in required_variables:
    assert variable_name in globals(), (
        f"{variable_name} n'existe pas. "
        "Relance les cellules précédentes du notebook, "
        "sans relancer l'entraînement."
    )

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUTPUT_DIR = Path(
    globals().get(
        "OUTPUT_DIR",
        "/kaggle/working/"
        "uniformerv2_output_partial_finetuning",
    )
)

BEST_MODEL_PATH = Path(
    globals().get(
        "BEST_MODEL_PATH",
        OUTPUT_DIR / "best_model.pth",
    )
)

LATEST_CHECKPOINT_PATH = Path(
    globals().get(
        "LATEST_CHECKPOINT_PATH",
        OUTPUT_DIR / "latest_checkpoint.pth",
    )
)

HISTORY_PATH = Path(
    globals().get(
        "HISTORY_PATH",
        OUTPUT_DIR / "history.json",
    )
)

assert BEST_MODEL_PATH.exists(), (
    f"Meilleur checkpoint introuvable : {BEST_MODEL_PATH}"
)

best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

assert "model_state_dict" in best_checkpoint

model.load_state_dict(
    best_checkpoint["model_state_dict"],
    strict=True,
)

model = model.to(DEVICE)
model.eval()

classes = best_checkpoint.get(
    "classes",
    globals().get("classes"),
)

assert classes is not None
assert len(classes) == 12
assert classes[7] == "normal"

NUM_CLASSES = len(classes)
NORMAL_CLASS_INDEX = classes.index("normal")

saved_epoch = best_checkpoint.get("epoch")
best_val_accuracy = best_checkpoint.get(
    "best_validation_accuracy"
)

best_val_macro_f1 = best_checkpoint.get(
    "best_validation_macro_f1"
)

print("=" * 70)
print("MEILLEUR MODÈLE UNIFORMERV2 CHARGÉ")
print("=" * 70)
print("Checkpoint :", BEST_MODEL_PATH)
print("Device     :", DEVICE)
print("Epoch      :", saved_epoch)

if best_val_macro_f1 is not None:
    print(
        "Meilleur validation macro F1 :",
        f"{best_val_macro_f1:.4f}",
    )

if best_val_accuracy is not None:
    print(
        "Meilleure validation accuracy :",
        f"{100 * best_val_accuracy:.2f}%",
    )

print("Nombre de classes :", NUM_CLASSES)
print("Classe normale     :", NORMAL_CLASS_INDEX, classes[7])

print("\n UNIFORMERV2 BEST PRÊT POUR L'INFÉRENCE")

In [ ]:
# ============================================================
# UF2 — PRÉDICTIONS VALIDATION ET TEST
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

ANNOTATIONS_DIR = Path(
    "/kaggle/working/annotations"
)

EVALUATION_DIR = Path(
    "/kaggle/working/uniformerv2_final_evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

AMP_ENABLED = DEVICE.type == "cuda"


def export_uniformerv2_predictions(
    loader,
    annotation_csv: Path,
    split_name: str,
):
    """
    Lance UniFormerV2 sur un split et crée un CSV standardisé
    compatible avec le notebook comparison_and_AT.
    """

    assert annotation_csv.exists(), annotation_csv

    annotations_df = pd.read_csv(
        annotation_csv,
        header=None,
        names=[
            "video_path",
            "annotation_label",
        ],
    )

    model.eval()

    all_probabilities = []
    all_true_labels = []

    print("\n" + "=" * 70)
    print(f"INFÉRENCE UNIFORMERV2 — {split_name.upper()}")
    print("=" * 70)

    with torch.inference_mode():

        for videos, labels in tqdm(
            loader,
            desc=split_name,
        ):
            videos = videos.to(
                DEVICE,
                non_blocking=True,
            )

            with torch.amp.autocast(
                device_type="cuda",
                enabled=AMP_ENABLED,
                dtype=torch.float16,
            ):
                logits = model(videos)

            # Certaines architectures peuvent renvoyer un tuple.
            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            assert logits.ndim == 2
            assert logits.shape[1] == NUM_CLASSES

            probabilities = torch.softmax(
                logits.float(),
                dim=1,
            )

            all_probabilities.append(
                probabilities.cpu()
            )

            all_true_labels.append(
                labels.cpu()
            )

    probabilities = torch.cat(
        all_probabilities,
        dim=0,
    ).numpy()

    true_labels = torch.cat(
        all_true_labels,
        dim=0,
    ).numpy().astype(int)

    predicted_labels = probabilities.argmax(
        axis=1
    ).astype(int)

    confidence = probabilities.max(
        axis=1
    )

    top5_labels = np.argsort(
        -probabilities,
        axis=1,
    )[:, :5]

    correct_top5 = np.array(
        [
            true_label in top5
            for true_label, top5
            in zip(true_labels, top5_labels)
        ],
        dtype=int,
    )

    # --------------------------------------------------------
    # Vérifier qu'il y a bien 335 vidéos et que l'ordre du
    # DataLoader correspond au fichier d'annotations.
    # --------------------------------------------------------

    print("Prédictions :", len(true_labels))
    print("Annotations :", len(annotations_df))

    assert len(true_labels) == 335, (
        f"{len(true_labels)} prédictions obtenues "
        "au lieu de 335."
    )

    assert len(annotations_df) == 335

    annotation_labels = annotations_df[
        "annotation_label"
    ].to_numpy(dtype=int)

    np.testing.assert_array_equal(
        true_labels,
        annotation_labels,
        err_msg=(
            "L'ordre du DataLoader ne correspond pas "
            "à l'ordre du fichier d'annotations."
        ),
    )

    # --------------------------------------------------------
    # Construire le CSV final
    # --------------------------------------------------------

    video_paths = annotations_df[
        "video_path"
    ].astype(str)

    predictions_df = pd.DataFrame(
        {
            "model": "UniFormerV2-B/16",
            "split": split_name,
            "video_id": [
                Path(path).stem
                for path in video_paths
            ],
            "video_path": video_paths,
            "true_label": true_labels,
            "true_class": [
                classes[index]
                for index in true_labels
            ],
            "predicted_label": predicted_labels,
            "predicted_class": [
                classes[index]
                for index in predicted_labels
            ],
            "confidence": confidence,
            "correct_top1": (
                predicted_labels == true_labels
            ).astype(int),
            "correct_top5": correct_top5,
            "view_count": 1,
        }
    )

    for class_index in range(NUM_CLASSES):
        predictions_df[
            f"prob_{class_index}"
        ] = probabilities[:, class_index]

    assert predictions_df[
        "confidence"
    ].between(0, 1).all()

    assert predictions_df[
        "video_path"
    ].nunique() == 335

    csv_path = (
        EVALUATION_DIR
        / f"uniformerv2_{split_name}_predictions.csv"
    )

    predictions_df.to_csv(
        csv_path,
        index=False,
    )

    # --------------------------------------------------------
    # Métriques
    # --------------------------------------------------------

    top1_accuracy = accuracy_score(
        true_labels,
        predicted_labels,
    )

    top5_accuracy = float(
        correct_top5.mean()
    )

    macro_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        true_labels,
        predicted_labels,
        average="weighted",
        zero_division=0,
    )

    abnormal_predicted_as_normal = int(
        (
            (true_labels != NORMAL_CLASS_INDEX)
            & (
                predicted_labels
                == NORMAL_CLASS_INDEX
            )
        ).sum()
    )

    report_dictionary = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
        target_names=classes,
        output_dict=True,
        zero_division=0,
    )

    report_text = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
        target_names=classes,
        digits=4,
        zero_division=0,
    )

    confusion = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=list(range(NUM_CLASSES)),
    )

    metrics = {
        "model": "UniFormerV2-B/16",
        "split": split_name,
        "number_of_videos": int(
            len(predictions_df)
        ),
        "top1_accuracy": float(
            top1_accuracy
        ),
        "top5_accuracy": float(
            top5_accuracy
        ),
        "macro_f1": float(
            macro_f1
        ),
        "weighted_f1": float(
            weighted_f1
        ),
        "normal_class_index": int(
            NORMAL_CLASS_INDEX
        ),
        "abnormal_predicted_as_normal": (
            abnormal_predicted_as_normal
        ),
    }

    metrics_path = (
        EVALUATION_DIR
        / f"uniformerv2_{split_name}_metrics.json"
    )

    metrics_path.write_text(
        json.dumps(
            metrics,
            indent=2,
        ),
        encoding="utf-8",
    )

    report_json_path = (
        EVALUATION_DIR
        / (
            f"uniformerv2_{split_name}_"
            "classification_report.json"
        )
    )

    report_json_path.write_text(
        json.dumps(
            report_dictionary,
            indent=2,
        ),
        encoding="utf-8",
    )

    report_txt_path = (
        EVALUATION_DIR
        / (
            f"uniformerv2_{split_name}_"
            "classification_report.txt"
        )
    )

    report_txt_path.write_text(
        report_text,
        encoding="utf-8",
    )

    confusion_csv_path = (
        EVALUATION_DIR
        / (
            f"uniformerv2_{split_name}_"
            "confusion_matrix.csv"
        )
    )

    pd.DataFrame(
        confusion,
        index=classes,
        columns=classes,
    ).to_csv(
        confusion_csv_path
    )

    print("\nRésultats :")
    print(
        f"Top-1       : "
        f"{100 * top1_accuracy:.3f}%"
    )
    print(
        f"Top-5       : "
        f"{100 * top5_accuracy:.3f}%"
    )
    print(
        f"Macro F1    : {macro_f1:.4f}"
    )
    print(
        f"Weighted F1 : {weighted_f1:.4f}"
    )
    print(
        "Anormales prédites normal :",
        abnormal_predicted_as_normal,
    )

    print("\nCSV créé :", csv_path)

    return predictions_df, metrics, confusion


# ============================================================
# VALIDATION
# ============================================================

(
    uniformerv2_validation_df,
    uniformerv2_validation_metrics,
    uniformerv2_validation_confusion,
) = export_uniformerv2_predictions(
    loader=val_loader,
    annotation_csv=(
        ANNOTATIONS_DIR / "val.csv"
    ),
    split_name="validation",
)


# ============================================================
# TEST
# ============================================================

(
    uniformerv2_test_df,
    uniformerv2_test_metrics,
    uniformerv2_test_confusion,
) = export_uniformerv2_predictions(
    loader=test_loader,
    annotation_csv=(
        ANNOTATIONS_DIR / "test.csv"
    ),
    split_name="test",
)

print(
    "\n LES DEUX CSV UNIFORMERV2 "
    "ONT ÉTÉ CRÉÉS"
)

In [ ]:
# ============================================================
# UF3 — MATRICES DE CONFUSION
# ============================================================

import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay,
)


def save_uniformerv2_confusion_matrix(
    confusion,
    split_name,
):
    figure, axis = plt.subplots(
        figsize=(13, 11)
    )

    display_object = ConfusionMatrixDisplay(
        confusion_matrix=confusion,
        display_labels=classes,
    )

    display_object.plot(
        ax=axis,
        xticks_rotation=45,
        values_format="d",
        colorbar=False,
    )

    axis.set_title(
        f"UniFormerV2-B/16 — {split_name}"
    )

    figure.tight_layout()

    output_path = (
        EVALUATION_DIR
        / (
            f"uniformerv2_{split_name}_"
            "confusion_matrix.png"
        )
    )

    figure.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()

    print("Sauvegardé :", output_path)


save_uniformerv2_confusion_matrix(
    uniformerv2_validation_confusion,
    "validation",
)

save_uniformerv2_confusion_matrix(
    uniformerv2_test_confusion,
    "test",
)

print("\n  MATRICES UNIFORMERV2 CRÉÉES")

In [ ]:
# ============================================================
# UF4 — RÉSUMÉ ET ARCHIVE À TÉLÉCHARGER
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

summary_df = pd.DataFrame(
    [
        uniformerv2_validation_metrics,
        uniformerv2_test_metrics,
    ]
)

summary_path = (
    EVALUATION_DIR
    / "uniformerv2_validation_test_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

display(summary_df)

# Copier l'historique dans le dossier léger.
if HISTORY_PATH.exists():
    shutil.copy2(
        HISTORY_PATH,
        EVALUATION_DIR / "history.json",
    )

archive_path = shutil.make_archive(
    base_name=(
        "/kaggle/working/"
        "uniformerv2_predictions_and_metrics"
    ),
    format="zip",
    root_dir=str(EVALUATION_DIR),
)

print("\nArchive créée :", archive_path)
print(
    "Taille :",
    round(
        Path(archive_path).stat().st_size
        / 1024**2,
        2,
    ),
    "MiB",
)

print("\nFichiers importants :")
print("1.", BEST_MODEL_PATH)
print("2.", LATEST_CHECKPOINT_PATH)
print("3.", HISTORY_PATH)
print("4.", archive_path)

print(
    "\n  TÉLÉCHARGE "
    "uniformerv2_predictions_and_metrics.zip"
)

In [ ]:
# ============================================================
# UF5 — SAUVEGARDE FINALE DANS UN DATASET KAGGLE
# À lancer seulement après UF4
# ============================================================

from pathlib import Path
import json
import os
import shutil

import kagglehub

UPLOAD_DIR = Path(
    "/kaggle/working/"
    "uniformerv2_final_dataset_upload"
)

DATASET_HANDLE = (
    "nisrineamalkhnifass/"
    "uniformerv2-final-rhad-results"
)

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)

UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_files = [
    BEST_MODEL_PATH,
    LATEST_CHECKPOINT_PATH,
    EVALUATION_DIR
    / "uniformerv2_validation_predictions.csv",
    EVALUATION_DIR
    / "uniformerv2_test_predictions.csv",
]

for required_file in required_files:
    assert required_file.exists(), (
        f"Fichier indispensable absent : "
        f"{required_file}"
    )


def add_file(
    source: Path,
    destination_name=None,
):
    destination = (
        UPLOAD_DIR
        / (
            destination_name
            if destination_name is not None
            else source.name
        )
    )

    try:
        os.link(
            source,
            destination,
        )
    except OSError:
        shutil.copy2(
            source,
            destination,
        )

    print(
        "Ajouté :",
        destination.name,
        f"({destination.stat().st_size / 1024**2:.2f} MiB)",
    )


add_file(
    BEST_MODEL_PATH,
    "uniformerv2_best_model.pth",
)

add_file(
    LATEST_CHECKPOINT_PATH,
    "uniformerv2_latest_checkpoint.pth",
)

if HISTORY_PATH.exists():
    add_file(HISTORY_PATH)

for source in sorted(
    EVALUATION_DIR.iterdir()
):
    if source.is_file():
        add_file(source)

for annotation_name in [
    "train.csv",
    "val.csv",
    "test.csv",
    "label_mapping.csv",
]:
    source = (
        ANNO_DIR / annotation_name
    )

    if source.exists():
        add_file(
            source,
            "annotations_" + annotation_name,
        )


summary = {
    "model": "UniFormerV2-B/16",
    "normal_class_index": 7,
    "best_checkpoint": (
        "uniformerv2_best_model.pth"
    ),
    "latest_checkpoint": (
        "uniformerv2_latest_checkpoint.pth"
    ),
    "files_for_comparison_and_AT": [
        "uniformerv2_validation_predictions.csv",
        "uniformerv2_test_predictions.csv",
    ],
}

(
    UPLOAD_DIR
    / "uniformerv2_dataset_manifest.json"
).write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 75)
print("UPLOAD UNIFORMERV2 FINAL VERS KAGGLE")
print("=" * 75)

kagglehub.dataset_upload(
    DATASET_HANDLE,
    str(UPLOAD_DIR),
    version_notes=(
        "UniFormerV2-B/16 RHAD final: "
        "best/latest checkpoints, exact split annotations, "
        "validation/test predictions, metrics and reports."
    ),
)

print(
    "\nUNIFORMERV2 EST SÉCURISÉ "
    "ET PRÊT POUR COMPARISON_AND_AT"
)


In [ ]:
# ============================================================
# UF6 — VÉRIFICATION FINALE DES FICHIERS PRÉDICTIFS
# À lancer après UF2, ou après UF5 pour tout vérifier
# ============================================================

from pathlib import Path
import pandas as pd

EVALUATION_DIR = Path(
    "/kaggle/working/uniformerv2_final_evaluation"
)

VALIDATION_PREDICTIONS_PATH = (
    EVALUATION_DIR
    / "uniformerv2_validation_predictions.csv"
)

TEST_PREDICTIONS_PATH = (
    EVALUATION_DIR
    / "uniformerv2_test_predictions.csv"
)

required_prediction_columns = {
    "model",
    "split",
    "video_id",
    "video_path",
    "true_label",
    "true_class",
    "predicted_label",
    "predicted_class",
    "confidence",
    "correct_top1",
    "correct_top5",
    "view_count",
}

required_probability_columns = {
    f"prob_{class_index}"
    for class_index in range(12)
}

print("=" * 75)
print("VÉRIFICATION DES PRÉDICTIONS UNIFORMERV2")
print("=" * 75)

for split_name, csv_path in [
    ("validation", VALIDATION_PREDICTIONS_PATH),
    ("test", TEST_PREDICTIONS_PATH),
]:
    assert csv_path.exists(), (
        f"Fichier absent : {csv_path}"
    )

    dataframe = pd.read_csv(csv_path)

    missing_columns = (
        required_prediction_columns
        | required_probability_columns
    ) - set(dataframe.columns)

    assert not missing_columns, (
        f"Colonnes manquantes dans {csv_path.name} : "
        f"{sorted(missing_columns)}"
    )

    assert len(dataframe) == 335, (
        f"{csv_path.name} contient {len(dataframe)} lignes "
        "au lieu de 335."
    )

    assert dataframe["video_path"].nunique() == 335
    assert dataframe["true_label"].between(0, 11).all()
    assert dataframe["predicted_label"].between(0, 11).all()
    assert dataframe["confidence"].between(0, 1).all()
    assert (dataframe["split"] == split_name).all()

    probability_sum = dataframe[
        sorted(
            required_probability_columns,
            key=lambda name: int(name.split("_")[1]),
        )
    ].sum(axis=1)

    assert probability_sum.between(
        0.999,
        1.001,
    ).all(), (
        f"Les probabilités de {csv_path.name} "
        "ne totalisent pas 1."
    )

    print(
        f"{split_name:<12} : "
        f"{len(dataframe)} vidéos | "
        f"Top-1 = {100 * dataframe['correct_top1'].mean():.3f}% | "
        f"Top-5 = {100 * dataframe['correct_top5'].mean():.3f}%"
    )
    print("  ", csv_path)

print("\nFichiers destinés à comparison_and_AT :")
print("1.", VALIDATION_PREDICTIONS_PATH)
print("2.", TEST_PREDICTIONS_PATH)

print(
    "\nNOTEBOOK UNIFORMERV2 COMPLET : "
    "LES DEUX FICHIERS PRÉDICTIFS SONT VALIDES"
)
